# KPI Calculations


## Final KPI function that calculates all KPI's at once as running modular function for each one and then merging them would be a time taking process

In [26]:
# Importing all the cleaned datasets.
import pandas as pd
df_balance_sheet = pd.read_csv(r"D:\Financial KPI Dashboard\data\processed\balance_clean.csv")
df_cash_flow = pd.read_csv(r'D:\Financial KPI Dashboard\data\processed\cashflow_clean.csv')
df_inc_stmt = pd.read_csv(r"D:\Financial KPI Dashboard\data\processed\income_clean.csv")

In [38]:
import pandas as pd

def calculate_all_kpis(df_inc, df_bal, df_cf):
    """
    Merges Income Statement, Balance Sheet, and Cash Flow statements.
    Calculates all required dashboard KPIs 
    """
    # 0. Drop redundant 'year' columns to prevent Pandas overlap errors
    df_bal_clean = df_bal.drop(columns=['year'], errors='ignore')
    df_cf_clean = df_cf.drop(columns=['year'], errors='ignore')
    
    # 1. Merge all three dataframes seamlessly using their shared Date index
    df_master = df_inc.join(df_bal_clean).join(df_cf_clean)
    
    # Calculate Total Equity (needed for ROE and Debt/Equity)
    df_master['total_equity'] = df_master['equity_capital'] + df_master['reserves']
    
    # Calculate EBIT proxy for the bank (Profit Before Tax + Interest Expense)
    ebit = df_master['profit_before_tax'] + df_master['interest']
    
    # ---------------------------------------------------------
    # 2. CALCULATE ALL KPIs (Multiplying by 100 for % columns)
    # ---------------------------------------------------------
    
    df_master['Revenue_Cr'] = df_master['revenue']
    df_master['Revenue_YoY_%'] = (df_master['revenue'].pct_change() * 100).round(2)
    
    df_master['EBITDA_Margin_%'] = ((ebit / df_master['revenue']) * 100).round(2)
    df_master['Net_Margin_%'] = ((df_master['net_profit'] / df_master['revenue']) * 100).round(2)
    
    df_master['ROE_%'] = ((df_master['net_profit'] / df_master['total_equity']) * 100).round(2)
    df_master['ROA_%'] = ((df_master['net_profit'] / df_master['total_assets']) * 100).round(2)
    
    df_master['Debt_Equity'] = (df_master['borrowing'] / df_master['total_equity']).round(2)
    
    # Interest Coverage = EBIT / Interest
    df_master['Interest_Coverage'] = (ebit / df_master['interest']).round(2)
    
    # Free Cash Flow Metrics
    df_master['FCF_Cr'] = df_master['free_cash_flow']
    df_master['FCF_Margin_%'] = ((df_master['free_cash_flow'] / df_master['revenue']) * 100).round(2)
    
    # ---------------------------------------------------------
    # 3. FILTER TO TARGET SCHEMA
    # ---------------------------------------------------------
    
    
    target_columns = [
        'Revenue_Cr', 'Revenue_YoY_%', 'EBITDA_Margin_%', 'Net_Margin_%',
        'ROE_%', 'ROA_%', 'Debt_Equity', 'Interest_Coverage', 
        'FCF_Cr', 'FCF_Margin_%'
    ]
    
    return df_master[target_columns]

# Execute the pipeline using your three cleaned dataframes
df_final_kpis = calculate_all_kpis(df_inc_stmt, df_balance_sheet, df_cash_flow)

# Drop the very first row (2015) because YoY% will be NaN
df_final_kpis = df_final_kpis.dropna()

# Display the final master table
print(df_final_kpis)

    Revenue_Cr  Revenue_YoY_%  EBITDA_Margin_%  Net_Margin_%  ROE_%  ROA_%  \
1        63162          24.66            84.83         20.30  17.25   1.68   
2        73271          16.00            83.85         20.90  16.69   1.72   
3        85288          16.40            83.07         21.76  16.94   1.68   
4       105161          23.30            83.71         21.34  14.61   1.74   
5       122189          16.19            82.11         22.34  15.48   1.73   
6       128552           5.21            79.38         24.78  15.18   1.77   
7       135936           5.74            80.52         28.07  15.43   1.80   
8       170754          25.61            81.57         27.03  15.94   1.82   
9       283649          66.12            81.34         23.07  14.34   1.62   
10      336367          18.59            83.28         21.83  14.07   1.67   
11      348615           3.64            82.51         22.72  13.62   1.61   

    Debt_Equity  Interest_Coverage  FCF_Cr  FCF_Margin_%  
1   

In [39]:
# Converting the master KPI table to csv and excel format for Dashboarding
df_final_kpis.to_csv(r'D:\Financial KPI Dashboard\dashboard\KPI_data.csv')
df_final_kpis.to_excel(r'D:\Financial KPI Dashboard\dashboard\KPI_data.xlsx')

### NOTE - Below are some modular functions which I created for calculating each KPI but it was getting hectic so I just created one function to calculate all KPIs together as we can merge all three datasets into one.

## 1. Revenue Grwoth (Year-on-Year) and Net Profit Margin

In [33]:
import pandas as pd

def calculate_profitability_kpis(df_income):
    """
    Calculates KPIs: Revenue Growth (YoY) and Net Profit Margin.
    """
    kpi_df = df_income.copy()
    
    # Revenue Growth (YoY)
    # .pct_change() automatically calculates the % difference from the previous row
    kpi_df['revenue_growth_yoy'] = kpi_df['revenue'].pct_change()
    
    # Net Profit Margin
    # Simple vectorized division across the entire dataframe
    kpi_df['net_profit_margin'] = kpi_df['net_profit'] / kpi_df['revenue']
    
    # Rounding to 4 decimal places (e.g., 0.1532 -> 15.32%) for cleaner downstream modeling
    kpi_df['revenue_growth_yoy'] = kpi_df['revenue_growth_yoy'].round(4)
    kpi_df['net_profit_margin'] = kpi_df['net_profit_margin'].round(4)
    
    return kpi_df

# Run the calculation on your clean dataframe
df_inc_stmt_revgrw = calculate_profitability_kpis(df_inc_stmt)
# Convert decimal to percent 
df_inc_stmt_revgrw['revenue_growth_yoy'] = df_inc_stmt_revgrw['revenue_growth_yoy']*100

# Verify the results by printing just the relevant columns
print(df_inc_stmt_revgrw[['revenue','revenue_growth_yoy','net_profit', 'net_profit_margin']].tail())

    revenue  revenue_growth_yoy  net_profit  net_profit_margin
7    135936                5.74       38151             0.2807
8    170754               25.61       46149             0.2703
9    283649               66.12       65446             0.2307
10   336367               18.59       73440             0.2183
11   348615                3.64       79219             0.2272


In [31]:
def calculate_revenue_cagr(df_income):
    """
    Calculates the Compound Annual Growth Rate (CAGR) for Revenue.
    Uses row count for time period to avoid datetime index errors.
    """
    # Extract the first and last revenue values
    initial_revenue = df_income['revenue'].iloc[0]
    final_revenue = df_income['revenue'].iloc[-1]
    
    # Calculate the time period (t) safely using row count
    # 12 rows of annual data = 11 years of growth (t=11)
    t = len(df_income) - 1
    
    # Apply the CAGR formula
    cagr = ((final_revenue / initial_revenue) ** (1 / t)) - 1
    
    return cagr

# Execute and format the output
revenue_cagr = calculate_revenue_cagr(df_inc_stmt_revgrw)

# Print it as a readable percentage
print(f"HDFC Bank Revenue CAGR: {revenue_cagr * 100:.2f}%")

HDFC Bank Revenue CAGR: 19.16%


In [34]:
def calculate_return_kpis(df_income, df_bal_sheet):
    """
    Merges the Income Statement and Balance Sheet on their date index 
    to calculate Return on Equity (ROE) and Return on Assets (ROA).
    """
    # 1. Bring in only the necessary columns from the Balance Sheet
    cols_needed = ['equity_capital', 'reserves', 'total_assets']
    
    # .join() safely matches the rows perfectly by their Date index!
    df_merged = df_income.join(df_bal_sheet[cols_needed])
    
    # 2. Calculate Total Shareholder Equity
    df_merged['total_equity'] = df_merged['equity_capital'] + df_merged['reserves']
    
    # 3. Calculate the Return KPIs
    df_merged['roe'] = df_merged['net_profit'] / df_merged['total_equity']
    df_merged['roa'] = df_merged['net_profit'] / df_merged['total_assets']
    
    # 4. Round to 4 decimal places (e.g., 0.1650 -> 16.5%)
    df_merged['roe'] = df_merged['roe'].round(4)
    df_merged['roa'] = df_merged['roa'].round(4)
    # 5. converting roa and roe to percentages
    df_merged['roa'] = df_merged['roa']*100
    df_merged['roe'] = df_merged['roe']*100
    return df_merged

# Execute the function using our clean dataframes
df_kpi_master = calculate_return_kpis(df_inc_stmt, df_balance_sheet)

# Print the results to verify the math
print(df_kpi_master[['net_profit', 'total_equity', 'total_assets', 'roe', 'roa']].tail())

    net_profit  total_equity  total_assets    roe   roa
7        38151        247327       2122934  15.43  1.80
8        46149        289438       2530432  15.94  1.82
9        65446        456396       4030194  14.34  1.62
10       73440        521789       4392110  14.07  1.67
11       79219        581514       4908041  13.62  1.61
